# Build Production Vector Store from Pre-built Parquet


In [1]:
import sys
sys.path.insert(0, '..')

import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
import pandas as pd
import chromadb
from pathlib import Path

PARQUET_PATH     = Path("../data/raw/complaint_embeddings.parquet")
VECTOR_STORE_DIR = Path("../vector_store")
COLLECTION_NAME  = "complaint_chunks"

print("OK")

OK


In [2]:
# Reads only the file footer — no data loaded into RAM
parquet_file = pq.ParquetFile(PARQUET_PATH)

print("Schema:")
print(parquet_file.schema)
print()
print(f"Total rows      : {parquet_file.metadata.num_rows:,}")
print(f"Total row groups: {parquet_file.metadata.num_row_groups}")

Schema:
required group field_id=-1 schema {
  optional binary field_id=-1 id (String);
  optional binary field_id=-1 document (String);
  optional group field_id=-1 embedding (List) {
    repeated group field_id=-1 list {
      optional double field_id=-1 element;
    }
  }
  optional group field_id=-1 metadata {
    optional int64 field_id=-1 chunk_index;
    optional binary field_id=-1 company (String);
    optional binary field_id=-1 complaint_id (String);
    optional binary field_id=-1 date_received (String);
    optional binary field_id=-1 issue (String);
    optional binary field_id=-1 product (String);
    optional binary field_id=-1 product_category (String);
    optional binary field_id=-1 state (String);
    optional binary field_id=-1 sub_issue (String);
    optional int64 field_id=-1 total_chunks;
  }
}


Total rows      : 1,375,327
Total row groups: 2


In [3]:
# Reads only first 100 rows — safe
first_batch = next(parquet_file.iter_batches(batch_size=100))

# Flatten struct columns using PyArrow BEFORE converting to pandas
# This turns metadata.complaint_id, metadata.product_category etc
# into flat columns — much faster than apply(pd.Series)
flat_table = pa.Table.from_batches([first_batch]).flatten()
df_peek    = flat_table.to_pandas()

print("Flattened columns:", df_peek.columns.tolist())
print()
print("Sample row (no embedding):")
print(df_peek.drop(columns=["embedding"]).iloc[0])

Flattened columns: ['id', 'document', 'embedding', 'metadata.chunk_index', 'metadata.company', 'metadata.complaint_id', 'metadata.date_received', 'metadata.issue', 'metadata.product', 'metadata.product_category', 'metadata.state', 'metadata.sub_issue', 'metadata.total_chunks']

Sample row (no embedding):
id                                                                  14069121_0
document                     a card was opened under my name by a fraudster...
metadata.chunk_index                                                         0
metadata.company                                                CITIBANK, N.A.
metadata.complaint_id                                                 14069121
metadata.date_received                                              2025-06-13
metadata.issue                                           Getting a credit card
metadata.product                                                   Credit card
metadata.product_category                                     

In [4]:
VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)
client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))

try:
    client.delete_collection(COLLECTION_NAME)
    print("Deleted old collection")
except Exception:
    pass

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

total_rows = parquet_file.metadata.num_rows
print(f"Collection ready. Rows to index: {total_rows:,}")

Deleted old collection
Collection ready. Rows to index: 1,375,327


In [5]:
import time

BATCH_SIZE = 1000  # lower to 500 if you still get memory errors

# After flatten(), metadata struct becomes columns named like:
# "metadata.complaint_id", "metadata.product_category", etc.
# We map them to clean names for ChromaDB storage.
META_FIELD_MAP = {
    "metadata.complaint_id"    : "complaint_id",
    "metadata.product_category": "product_category",
    "metadata.product"         : "product",
    "metadata.issue"           : "issue",
    "metadata.sub_issue"       : "sub_issue",
    "metadata.company"         : "company",
    "metadata.state"           : "state",
    "metadata.date_received"   : "date_received",
    "metadata.chunk_index"     : "chunk_index",
    "metadata.total_chunks"    : "total_chunks",
}

processed  = 0
start_time = time.time()

for batch in parquet_file.iter_batches(batch_size=BATCH_SIZE):

    # Flatten struct → flat column names like "metadata.complaint_id"
    flat_table = pa.Table.from_batches([batch]).flatten()
    df         = flat_table.to_pandas()

    ids        = df["id"].tolist()
    documents  = df["document"].fillna("").tolist()
    embeddings = [
        row.tolist() if hasattr(row, "tolist") else list(row)
        for row in df["embedding"]
    ]

    metadatas = []
    for _, row in df.iterrows():
        meta = {}
        for flat_col, clean_name in META_FIELD_MAP.items():
            if flat_col not in df.columns:
                continue
            val = row[flat_col]
            if val is None:
                continue
            # Skip NaN floats
            if isinstance(val, float) and np.isnan(val):
                continue
            # ChromaDB requires str / int / float — no numpy types
            if isinstance(val, (np.integer,)):
                val = int(val)
            elif isinstance(val, (np.floating,)):
                val = float(val)
            else:
                val = str(val)
            meta[clean_name] = val
        metadatas.append(meta)

    collection.upsert(
        ids=ids,
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas,
    )

    processed += len(df)
    elapsed    = time.time() - start_time
    rate       = processed / elapsed if elapsed > 0 else 1
    eta        = (total_rows - processed) / rate

    print(
        f"  {processed:>9,} / {total_rows:,}  |  "
        f"{elapsed/60:5.1f} min elapsed  |  "
        f"~{eta/60:5.1f} min left",
        end="\r",
    )

print(f"\n\n✓ Done in {(time.time()-start_time)/60:.1f} min")
print(f"  Chunks indexed: {collection.count():,}")
print(f"  Location: {VECTOR_STORE_DIR.resolve()}")

  1,375,327 / 1,375,327  |  186.2 min elapsed  |  ~  0.0 min left

✓ Done in 186.2 min
  Chunks indexed: 1,375,327
  Location: C:\Users\gtta1\OneDrive\Documents\KAIM\w7\rag-complaint-chatbot\vector_store


In [6]:
from src.embeddings import embed_texts

client2     = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))
collection2 = client2.get_collection(COLLECTION_NAME)
print(f"Reloaded: {collection2.count():,} chunks\n")

q_emb   = embed_texts(["credit card billing dispute"], show_progress=False)
results = collection2.query(
    query_embeddings=q_emb.tolist(),
    n_results=3,
    include=["documents", "metadatas", "distances"],
)

for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
), 1):
    print(f"Result {i} | {meta.get('product_category','?')} | dist {dist:.4f}")
    print(f"  {doc[:150]}...")
    print()

print("✓ Vector store ready — run app.py")

Reloaded: 1,375,327 chunks

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded.
Result 1 | Credit Card | dist 0.2217
  the credit card company is not handling my dispute properly. i reasserted the dispute on and sent documentation to show improper billing service not p...

Result 2 | Credit Card | dist 0.2383
  w they arrived at that decision was not explained. as monthly bills came in, there were late fees and interest charges, so i continued to call to prot...

Result 3 | Credit Card | dist 0.2401
  estroy the card s you may have for the account s . heres how we made this decision we made this decision for the following reason s you were previousl...

✓ Vector store ready — run app.py
